<a href="https://colab.research.google.com/github/jhysnvdl9/Jhysnvdl/blob/main/JhaysonVidal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

import random, time, numpy as np, torch
import matplotlib.pyplot as plt
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# 1. Reproducibility & Device Setup
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# 2. Data Pipeline
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_data = datasets.FashionMNIST(root='data', train=True, download=True, transform=transform)
test_data = datasets.FashionMNIST(root='data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=256, shuffle=False)

class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

# Display Sample Grid
images, labels = next(iter(train_loader))
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    img = images[i].squeeze() * 0.5 + 0.5  # Unnormalize
    ax.imshow(img, cmap='gray')
    ax.set_title(class_names[labels[i]])
    ax.axis('off')
plt.suptitle('Fashion-MNIST Sample Grid')
plt.show()

# 3. Base Model Architecture Definition
class FashionMLP(nn.Module):
    def __init__(self, hidden1=128, hidden2=64, activation=nn.ReLU):
        super().__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(784, hidden1)
        self.fc2 = nn.Linear(hidden1, hidden2)
        self.fc3 = nn.Linear(hidden2, 10)
        self.act = activation()

    def forward(self, x):
        x = self.flatten(x)
        x = self.act(self.fc1(x))
        x = self.act(self.fc2(x))
        logits = self.fc3(x)
        return logits

# Instantiate Base Model and Print Verification Outputs
base_model = FashionMLP().to(device)
print("\n--- Model Architecture ---")
print(base_model)

total_params = sum(p.numel() for p in base_model.parameters() if p.requires_grad)
print(f"\nTotal Trainable Parameters: {total_params:,}")

# Shape Verification Trace
dummy_input = torch.randn(64, 1, 28, 28).to(device)
print("\n--- Forward Pass Tensor Shapes ---")
print(f"Input Shape:    {dummy_input.shape}")
x = base_model.flatten(dummy_input)
print(f"After Flatten:  {x.shape}")
x = base_model.act(base_model.fc1(x))
print(f"After Layer 1:  {x.shape}")
x = base_model.act(base_model.fc2(x))
print(f"After Layer 2:  {x.shape}")
logits = base_model.fc3(x)
print(f"After Layer 3:  {logits.shape}")